<a href="https://colab.research.google.com/github/swirita/salmonellosis-forecasting-analysis/blob/main/notebooks/01_preparing_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prepare Salmonellosis Weekly Data for Analysis

* **Original Dataset Source:** [CDC NNDSS Weekly Data](https://data.cdc.gov/NNDSS/NNDSS-Weekly-Data/x9gk-5huc/about_data)
* **Data Last Updated:** September 10, 2026

> This notebook will load and prepare the weekly Salmonellosis data published by the Centers for Disease Control. The dataset contains only **Salmonellosis (excluding Salmonella Typhi infection and Salmonella Paratyphi infection)**.

## Data Flags

Some cells contain letters or symbols instead of numbers. These values explain the status of the reported data:
- **U - Unavailable:** The data was not available or could not be reported.
- **- - No Reported Cases:** No cases were reported for that disease.
- **N - Not Reportable:** The disease did not need to be reported in that area.
- **NN - Not Nationally Notifiable:** The disease was not required to be reported at the national level.
- **NP - Not Published:** The disease was reportable, but its data was not published.
- **NC - Not Calculated:** There was not enough data to calculate the value.
- **Cum - Cumulative:** The total number of cases from the beginning of the year until the current week.
- **Max - Maximum:** The highest weekly number of cases during the previous 52 weeks.


In [235]:
# Imports
import pandas as pd
import os
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

from sklearn import set_config

set_config(transform_output="pandas")

In [236]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [237]:
# Read the first worksheet directly from Google Sheets
sheet_id = "1FpPgAn6V-8UAkS4klZSe0yECrS6DJXR5QbCuwINShL0"
sheet_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(sheet_url)

print(f"Data loaded successfully: {df.shape[0]:,} rows and {df.shape[1]} columns.")


Data loaded successfully: 17,080 rows and 16 columns.


In [238]:
# Show tha Data
df.head()

,Reporting Area,Current MMWR Year,MMWR WEEK,Label,Current week,"Current week, flag",Previous 52 week Max,"Previous 52 weeks Max, flag",Cumulative YTD Current MMWR Year,"Cumulative YTD Current MMWR Year, flag",Cumulative YTD Previous MMWR Year,"Cumulative YTD Previous MMWR Year, flag",LOCATION1,LOCATION2,sort_order,geocode
0,US RESIDENTS,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,96,-,"1,212",-,96,-,333,-,NaN,US RESIDENTS,20220105601,NaN
1,NEW ENGLAND,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,2,-,82,-,2,-,24,-,NaN,NEW ENGLAND,20220105602,NaN
2,CONNECTICUT,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,NaN,-,19,-,NaN,-,3,-,CONNECTICUT,NaN,20220105603,POINT (-72.738288 41.575155)
3,MAINE,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,NaN,-,10,-,NaN,-,1,-,MAINE,NaN,20220105604,POINT (-69.06137 45.117911)
4,MASSACHUSETTS,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,2,-,44,-,2,-,15,-,MASSACHUSETTS,NaN,20220105605,POINT (-71.481104 42.151077)


In [239]:
# Check the dataset size and basic information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17080 entries, 0 to 17079
Data columns (total 16 columns):
 #   Column                                   Non-Null Count  Dtype 
---  ------                                   --------------  ----- 
 0   Reporting Area                           17080 non-null  object
 1   Current MMWR Year                        17080 non-null  int64 
 2   MMWR WEEK                                17080 non-null  int64 
 3   Label                                    17080 non-null  object
 4   Current week                             9923 non-null   object
 5   Current week, flag                       13401 non-null  object
 6   Previous 52 week Max                     17074 non-null  object
 7   Previous 52 weeks Max, flag              10853 non-null  object
 8   Cumulative YTD Current MMWR Year         15358 non-null  object
 9   Cumulative YTD Current MMWR Year, flag   11444 non-null  object
 10  Cumulative YTD Previous MMWR Year        16244 non-null  o

# Preprocessing the data

## Missing Values

In [240]:
# Check missing values in each column
df.isna().sum()

,0
Reporting Area,0
Current MMWR Year,0
MMWR WEEK,0
Label,0
Current week,7157
"Current week, flag",3679
Previous 52 week Max,6
"Previous 52 weeks Max, flag",6227
Cumulative YTD Current MMWR Year,1722
"Cumulative YTD Current MMWR Year, flag",5636


### Check Current Week Flags

In [241]:
# Check the values and counts in the Current week flag column
df["Current week, flag"].value_counts(dropna=False)

,count
"Current week, flag",
-,13400
NaN,3679
U,1


In [242]:
# Compare missing Current week values with their flags
missing_with_dash = df[df["Current week"].isna() &(df["Current week, flag"] == "-")].shape[0]

missing_with_U = df[df["Current week"].isna() &(df["Current week, flag"] == "U")].shape[0]

missing_with_no_flag = df[df["Current week"].isna() &(df["Current week, flag"].isna())].shape[0]

print("Missing Current week with '-' flag:", missing_with_dash)
print("Missing Current week with 'U' flag:", missing_with_U)
print("Missing Current week with no flag:", missing_with_no_flag)

Missing Current week with '-' flag: 7156
Missing Current week with 'U' flag: 1
Missing Current week with no flag: 0


In [243]:
# Replace unreported cases with zero
df.loc[df["Current week"].isna() &(df["Current week, flag"] == "-"),"Current week"] = 0

# Remove the one row with unavailable data
df = df[df["Current week, flag"] != "U"].copy()

# Convert Current week to numeric
df["Current week"] = pd.to_numeric(df["Current week"].astype(str).str.replace(",", ""))

# Check the result
print("Number of rows:", df.shape[0])
print("Missing Current week values:", df["Current week"].isna().sum())
print("Current week data type:", df["Current week"].dtype)

Number of rows: 17079
Missing Current week values: 0
Current week data type: int64


### Check Previous 52 Week Max Flags

In [244]:
# Check Previous 52 week Max flags
df["Previous 52 weeks Max, flag"].value_counts(dropna=False)

,count
"Previous 52 weeks Max, flag",
-,10847
NaN,6227
NC,5


In [245]:
# Missing values in previous 52 week Max = NC Flag impute them using SimpleImputer
df["Previous 52 week Max"].isna().sum()

np.int64(5)

In [246]:
# Convert Previous 52 week Max to numeric
df["Previous 52 week Max"] = pd.to_numeric(df["Previous 52 week Max"].astype(str).str.replace(",", ""),errors="coerce")

### Check Current Year Cumulative Flags

In [247]:
# Check Current Year Cumulative flags
df["Cumulative YTD Current MMWR Year, flag"].value_counts(dropna=False)

,count
"Cumulative YTD Current MMWR Year, flag",
-,11443
NaN,5636


In [248]:
df[df["Cumulative YTD Current MMWR Year"].isna()]["Cumulative YTD Current MMWR Year, flag"].value_counts(dropna=False)

,count
"Cumulative YTD Current MMWR Year, flag",
-,1721


In [249]:
# Replace missing cumulative values marked with "-" by zero
df.loc[df["Cumulative YTD Current MMWR Year"].isna() &(df["Cumulative YTD Current MMWR Year, flag"] == "-"),"Cumulative YTD Current MMWR Year"] = 0

In [250]:
# Check the result
df["Cumulative YTD Current MMWR Year"].isna().sum()

np.int64(0)

In [251]:
# Convert Cumulative YTD Current MMWR Year
df["Cumulative YTD Current MMWR Year"] = pd.to_numeric(df["Cumulative YTD Current MMWR Year"].astype(str).str.replace(",", ""))

### Check Cumulative YTD Previous Year

In [252]:
# Check Previous Year Cumulative flags
df["Cumulative YTD Previous MMWR Year, flag"].value_counts(dropna=False)

,count
"Cumulative YTD Previous MMWR Year, flag",
-,11126
NaN,5948
U,5


In [253]:
# Check flags only for missing Previous Year Cumulative values
df[df["Cumulative YTD Previous MMWR Year"].isna()]["Cumulative YTD Previous MMWR Year, flag"].value_counts(dropna=False)

,count
"Cumulative YTD Previous MMWR Year, flag",
-,830
U,5


In [254]:
# Replace missing values marked with "-" by zero
df.loc[df["Cumulative YTD Previous MMWR Year"].isna() &(df["Cumulative YTD Previous MMWR Year, flag"] == "-"),"Cumulative YTD Previous MMWR Year"] = 0

In [255]:
# Check remaining missing values
df["Cumulative YTD Previous MMWR Year"].isna().sum()

np.int64(5)

In [256]:
df.isna().sum()

,0
Reporting Area,0
Current MMWR Year,0
MMWR WEEK,0
Label,0
Current week,0
"Current week, flag",3679
Previous 52 week Max,5
"Previous 52 weeks Max, flag",6227
Cumulative YTD Current MMWR Year,0
"Cumulative YTD Current MMWR Year, flag",5636


In [257]:
# Convert Cumulative YTD Previous MMWR Year
df["Cumulative YTD Previous MMWR Year"] = pd.to_numeric(df["Cumulative YTD Previous MMWR Year"].astype(str).str.replace(",", ""),errors="coerce")

### Check Location

In [258]:
# Check whether LOCATION1 and LOCATION2 complete each other
both_missing = df[df["LOCATION1"].isna() & df["LOCATION2"].isna()].shape[0]
both_available = df[df["LOCATION1"].notna() & df["LOCATION2"].notna()].shape[0]
only_location1 = df[df["LOCATION1"].notna() & df["LOCATION2"].isna()].shape[0]
only_location2 = df[df["LOCATION1"].isna() & df["LOCATION2"].notna()].shape[0]
print("Both missing:", both_missing)
print("Both available:", both_available)
print("Only LOCATION1 available:", only_location1)
print("Only LOCATION2 available:", only_location2)

Both missing: 0
Both available: 0
Only LOCATION1 available: 13907
Only LOCATION2 available: 3172


In [259]:
# Combine LOCATION1 and LOCATION2 into one column
df["Location"] = df["LOCATION1"].fillna(df["LOCATION2"])
df[["Reporting Area", "LOCATION1", "LOCATION2", "Location"]].head(10)

,Reporting Area,LOCATION1,LOCATION2,Location
0,US RESIDENTS,NaN,US RESIDENTS,US RESIDENTS
1,NEW ENGLAND,NaN,NEW ENGLAND,NEW ENGLAND
2,CONNECTICUT,CONNECTICUT,NaN,CONNECTICUT
3,MAINE,MAINE,NaN,MAINE
4,MASSACHUSETTS,MASSACHUSETTS,NaN,MASSACHUSETTS
5,NEW HAMPSHIRE,NEW HAMPSHIRE,NaN,NEW HAMPSHIRE
6,RHODE ISLAND,RHODE ISLAND,NaN,RHODE ISLAND
7,VERMONT,VERMONT,NaN,VERMONT
8,MIDDLE ATLANTIC,NaN,MIDDLE ATLANTIC,MIDDLE ATLANTIC
9,NEW JERSEY,NEW JERSEY,NaN,NEW JERSEY


In [260]:
df['Location'].isna().sum()

np.int64(0)

In [261]:
# Compare Reporting Area with Combined Location
(df["Reporting Area"] == df["Location"]).value_counts()

,count
True,17079


In [262]:
# Drop Locations because Reporting Area = Location
df.drop(columns=["LOCATION1", "LOCATION2", "Location"],inplace=True)

In [263]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17079 entries, 0 to 17079
Data columns (total 14 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Reporting Area                           17079 non-null  object 
 1   Current MMWR Year                        17079 non-null  int64  
 2   MMWR WEEK                                17079 non-null  int64  
 3   Label                                    17079 non-null  object 
 4   Current week                             17079 non-null  int64  
 5   Current week, flag                       13400 non-null  object 
 6   Previous 52 week Max                     17074 non-null  float64
 7   Previous 52 weeks Max, flag              10852 non-null  object 
 8   Cumulative YTD Current MMWR Year         17079 non-null  int64  
 9   Cumulative YTD Current MMWR Year, flag   11443 non-null  object 
 10  Cumulative YTD Previous MMWR Year        17074 non-

### Check Missing Geocode Values

In [264]:
# Check which reporting areas have missing geocode values
df[df["geocode"].isna()]["Reporting Area"].value_counts()

,count
Reporting Area,
US RESIDENTS,156
NEW ENGLAND,156
MIDDLE ATLANTIC,156
EAST NORTH CENTRAL,156
NON-US RESIDENTS,156
WEST NORTH CENTRAL,156
EAST SOUTH CENTRAL,156
WEST SOUTH CENTRAL,156
PACIFIC,156


In [265]:
# Drop geocode because Reporting Area already identifies the location
df.drop(columns=["geocode"], inplace=True)

## Remove Unnecessary Columns

In [266]:
# Remove the website sorting column
df.drop(columns=["sort_order"], inplace=True)

In [267]:
# Remove Flag Columns after using them
flag_columns = ["Current week, flag","Previous 52 weeks Max, flag",
    "Cumulative YTD Current MMWR Year, flag","Cumulative YTD Previous MMWR Year, flag"]
df = df.drop(columns=flag_columns)

In [268]:
df.isna().sum()

,0
Reporting Area,0
Current MMWR Year,0
MMWR WEEK,0
Label,0
Current week,0
Previous 52 week Max,5
Cumulative YTD Current MMWR Year,0
Cumulative YTD Previous MMWR Year,5


## Duplicated Values

In [269]:
df.duplicated().sum()

np.int64(0)

## Dataset Coverage

In [270]:
# Check the dataset coverage
# check years and number of years
print("Years:",df["Current MMWR Year"].unique())
print("Number of years:",df["Current MMWR Year"].nunique())

# Check number of weeks
print("Weeks:",df["MMWR WEEK"].unique())
print("Number of weeks:",df["MMWR WEEK"].nunique())

# Check number of reporting areas
print("Number of reporting areas:",df["Reporting Area"].nunique())


Years: [2022 2023 2024 2025 2026]
Number of years: 5
Weeks: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50 51 52 53]
Number of weeks: 53
Number of reporting areas: 140


## Clean Reporting Area Names

In [271]:
# Convert Reporting Area names to uppercase
df["Reporting Area"] = df["Reporting Area"].str.upper()
df["Reporting Area"].nunique()

74

In [272]:
df["Reporting Area"].unique()

array(['US RESIDENTS', 'NEW ENGLAND', 'CONNECTICUT', 'MAINE',
       'MASSACHUSETTS', 'NEW HAMPSHIRE', 'RHODE ISLAND', 'VERMONT',
       'MIDDLE ATLANTIC', 'NEW JERSEY', 'NEW YORK', 'NEW YORK CITY',
       'PENNSYLVANIA', 'EAST NORTH CENTRAL', 'ILLINOIS', 'INDIANA',
       'MICHIGAN', 'OHIO', 'WISCONSIN', 'WEST NORTH CENTRAL', 'IOWA',
       'KANSAS', 'MINNESOTA', 'MISSOURI', 'NEBRASKA', 'NORTH DAKOTA',
       'SOUTH DAKOTA', 'SOUTH ATLANTIC', 'DELAWARE',
       'DISTRICT OF COLUMBIA', 'FLORIDA', 'GEORGIA', 'MARYLAND',
       'NORTH CAROLINA', 'SOUTH CAROLINA', 'VIRGINIA', 'WEST VIRGINIA',
       'EAST SOUTH CENTRAL', 'ALABAMA', 'KENTUCKY', 'MISSISSIPPI',
       'TENNESSEE', 'WEST SOUTH CENTRAL', 'ARKANSAS', 'LOUISIANA',
       'OKLAHOMA', 'TEXAS', 'MOUNTAIN', 'ARIZONA', 'COLORADO', 'IDAHO',
       'MONTANA', 'NEVADA', 'NEW MEXICO', 'UTAH', 'WYOMING', 'PACIFIC',
       'ALASKA', 'CALIFORNIA', 'HAWAII', 'OREGON', 'WASHINGTON',
       'US TERRITORIES', 'AMERICAN SAMOA', 'NORTHERN MARIANA

In [273]:
# Replace different names for the same reporting area
df["Reporting Area"] = df["Reporting Area"].replace({
    "U.S. RESIDENTS": "US RESIDENTS",
    "U.S. TERRITORIES": "US TERRITORIES",
    "NON-U.S. RESIDENTS": "NON-US RESIDENTS",
    "U.S. VIRGIN ISLANDS": "US VIRGIN ISLANDS",
    "COMMONWEALTH OF NORTHERN MARIANA ISLANDS": "NORTHERN MARIANA ISLANDS"
})

In [274]:
print('Number of reporting areas:',df["Reporting Area"].nunique())
df["Reporting Area"].unique()

Number of reporting areas: 70


array(['US RESIDENTS', 'NEW ENGLAND', 'CONNECTICUT', 'MAINE',
       'MASSACHUSETTS', 'NEW HAMPSHIRE', 'RHODE ISLAND', 'VERMONT',
       'MIDDLE ATLANTIC', 'NEW JERSEY', 'NEW YORK', 'NEW YORK CITY',
       'PENNSYLVANIA', 'EAST NORTH CENTRAL', 'ILLINOIS', 'INDIANA',
       'MICHIGAN', 'OHIO', 'WISCONSIN', 'WEST NORTH CENTRAL', 'IOWA',
       'KANSAS', 'MINNESOTA', 'MISSOURI', 'NEBRASKA', 'NORTH DAKOTA',
       'SOUTH DAKOTA', 'SOUTH ATLANTIC', 'DELAWARE',
       'DISTRICT OF COLUMBIA', 'FLORIDA', 'GEORGIA', 'MARYLAND',
       'NORTH CAROLINA', 'SOUTH CAROLINA', 'VIRGINIA', 'WEST VIRGINIA',
       'EAST SOUTH CENTRAL', 'ALABAMA', 'KENTUCKY', 'MISSISSIPPI',
       'TENNESSEE', 'WEST SOUTH CENTRAL', 'ARKANSAS', 'LOUISIANA',
       'OKLAHOMA', 'TEXAS', 'MOUNTAIN', 'ARIZONA', 'COLORADO', 'IDAHO',
       'MONTANA', 'NEVADA', 'NEW MEXICO', 'UTAH', 'WYOMING', 'PACIFIC',
       'ALASKA', 'CALIFORNIA', 'HAWAII', 'OREGON', 'WASHINGTON',
       'US TERRITORIES', 'AMERICAN SAMOA', 'NORTHERN MARIANA

## Sort Data by Reporting Area and Time
Area --> Year --> Week

In [275]:
df = df.sort_values(["Reporting Area","Current MMWR Year","MMWR WEEK"])
df.head(10)

,Reporting Area,Current MMWR Year,MMWR WEEK,Label,Current week,Previous 52 week Max,Cumulative YTD Current MMWR Year,Cumulative YTD Previous MMWR Year
38,ALABAMA,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,10,38.0,10,6.0
108,ALABAMA,2022,2,Salmonellosis (excluding Salmonella Typhi infe...,3,38.0,18,12.0
178,ALABAMA,2022,3,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,24,17.0
248,ALABAMA,2022,4,Salmonellosis (excluding Salmonella Typhi infe...,2,38.0,26,25.0
318,ALABAMA,2022,5,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,29,30.0
388,ALABAMA,2022,6,Salmonellosis (excluding Salmonella Typhi infe...,2,38.0,33,35.0
458,ALABAMA,2022,7,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,34,37.0
528,ALABAMA,2022,8,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,34,45.0
598,ALABAMA,2022,9,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,34,49.0
668,ALABAMA,2022,10,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,41,51.0


# Feature Engineering

In [276]:
# Get Seasons
def get_season(week):
    if week <= 13:
        return "Winter"
    elif week <= 26:
        return "Spring"
    elif week <= 39:
        return "Summer"
    else:
        return "Fall"
df["Season"] = df["MMWR WEEK"].apply(get_season)

In [277]:
# Get Previous week
df["Previous week cases"] = df.groupby("Reporting Area")["Current week"].shift(1)

In [278]:
df.head()

,Reporting Area,Current MMWR Year,MMWR WEEK,Label,Current week,Previous 52 week Max,Cumulative YTD Current MMWR Year,Cumulative YTD Previous MMWR Year,Season,Previous week cases
38,ALABAMA,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,10,38.0,10,6.0,Winter,NaN
108,ALABAMA,2022,2,Salmonellosis (excluding Salmonella Typhi infe...,3,38.0,18,12.0,Winter,10.0
178,ALABAMA,2022,3,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,24,17.0,Winter,3.0
248,ALABAMA,2022,4,Salmonellosis (excluding Salmonella Typhi infe...,2,38.0,26,25.0,Winter,0.0
318,ALABAMA,2022,5,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,29,30.0,Winter,2.0


In [279]:
df.isna().sum()

,0
Reporting Area,0
Current MMWR Year,0
MMWR WEEK,0
Label,0
Current week,0
Previous 52 week Max,5
Cumulative YTD Current MMWR Year,0
Cumulative YTD Previous MMWR Year,5
Season,0
Previous week cases,70


## Pipelines

In [280]:
# Numeric Features
numeric_features = ["Current MMWR Year","MMWR WEEK",
    "Previous 52 week Max","Cumulative YTD Previous MMWR Year",
    "Previous week cases"]
# Categorical Features
categorical_features = ["Reporting Area","Season"]

In [281]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [282]:
# Impute the selected features with the preprocessing pipeline
df_processed = preprocessor.fit_transform(df)

# Start with the full cleaned dataset so no original columns are lost
df_preprocessed = df.copy()

# Replace the selected columns with their imputed values
for column in numeric_features:
    df_preprocessed[column] = df_processed[f"numeric__{column}"]

for column in categorical_features:
    df_preprocessed[column] = df_processed[f"categorical__{column}"]

print(f"Preprocessed dataset shape: {df_preprocessed.shape[0]:,} rows and {df_preprocessed.shape[1]} columns.")

Preprocessed dataset shape: 17,079 rows and 10 columns.


In [283]:
# Confirm that the final dataset contains no missing values
df_preprocessed.isna().sum()

,0
Reporting Area,0
Current MMWR Year,0
MMWR WEEK,0
Label,0
Current week,0
Previous 52 week Max,0
Cumulative YTD Current MMWR Year,0
Cumulative YTD Previous MMWR Year,0
Season,0
Previous week cases,0


In [284]:
# Convert year and week back to integers
df_preprocessed["Current MMWR Year"] = (
    df_preprocessed["Current MMWR Year"].astype("int64")
)

df_preprocessed["MMWR WEEK"] = (
    df_preprocessed["MMWR WEEK"].astype("int64")
)

# Preview the complete dataset with the imputed values
df_preprocessed

,Reporting Area,Current MMWR Year,MMWR WEEK,Label,Current week,Previous 52 week Max,Cumulative YTD Current MMWR Year,Cumulative YTD Previous MMWR Year,Season,Previous week cases
38,ALABAMA,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,10,38.0,10,6.0,Winter,2.0
108,ALABAMA,2022,2,Salmonellosis (excluding Salmonella Typhi infe...,3,38.0,18,12.0,Winter,10.0
178,ALABAMA,2022,3,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,24,17.0,Winter,3.0
248,ALABAMA,2022,4,Salmonellosis (excluding Salmonella Typhi infe...,2,38.0,26,25.0,Winter,0.0
318,ALABAMA,2022,5,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,29,30.0,Winter,2.0
...,...,...,...,...,...,...,...,...,...,...
16785,WYOMING,2026,31,Salmonellosis (excluding Salmonella Typhi infe...,10,10.0,73,75.0,Summer,3.0
16855,WYOMING,2026,32,Salmonellosis (excluding Salmonella Typhi infe...,1,11.0,75,79.0,Summer,10.0
16925,WYOMING,2026,33,Salmonellosis (excluding Salmonella Typhi infe...,2,11.0,81,81.0,Summer,1.0
16995,WYOMING,2026,34,Salmonellosis (excluding Salmonella Typhi infe...,4,11.0,88,87.0,Summer,2.0


# Save the Cleaned Data to Google Drive

The cleaned dataset is saved as a CSV file in:

`My Drive/Salmonellosis Forecasting and Analysis/data/`

In [285]:
# Create the project folder and its data subfolder
project_folder = "/content/drive/MyDrive/Salmonellosis Forecasting and Analysis"
data_folder = os.path.join(project_folder, "data")

os.makedirs(data_folder, exist_ok=True)

print("Folder ready:", data_folder)

Folder ready: /content/drive/MyDrive/Salmonellosis Forecasting and Analysis/data


In [286]:
# Save the cleaned dataset without the pandas index
cleaned_file_path = os.path.join(data_folder, "salmonellosis_weekly_cleaned.csv")
df_preprocessed.to_csv(cleaned_file_path, index=False)

print("Cleaned data saved successfully.")
print("File location:", cleaned_file_path)
print(f"Saved dataset shape: {df_preprocessed.shape[0]:,} rows and {df_preprocessed.shape[1]} columns.")

Cleaned data saved successfully.
File location: /content/drive/MyDrive/Salmonellosis Forecasting and Analysis/data/salmonellosis_weekly_cleaned.csv
Saved dataset shape: 17,079 rows and 10 columns.
